In [51]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import re

In [52]:
def get_player_bio(player_id):
    url = f"https://www.fotmob.com/en/players/{player_id}"

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--headless=new")

    # options.add_argument(
    #     "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    #     "AppleWebKit/537.36 (KHTML, like Gecko) "
    #     "Chrome/124.0.0.0 Safari/537.36"
    # )

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    # wait for page to load (simple version)
    time.sleep(5)
    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    player_name = soup.find("h1").text.strip()
    print(player_name)

    bio = soup.find("div", class_="css-nlprju-PlayerBioCSS e1sx8s6x0")
    stat_blocks = bio.find_all("div", class_="css-to3w1c-StatValueCSS e1e6xf3b2")
    rows = []
    for block in stat_blocks:
        value = block.get_text(strip=True)
        parent = block.find_parent()
        title_tag = parent.find("div", class_="css-tp32vr-StatTitleCSS e1e6xf3b1")
        title = title_tag.get_text(strip=True) if title_tag else None
        rows.append({
            "stat": title,
            "value": value
        })

    position_block = soup.find("div", class_="css-1y1g69o-PositionSectionCSS e1sqdo1t7")
    position_tag = position_block.find("div", class_="css-1g41csj-PositionsCSS e1sqdo1t6")
    position = position_tag.get_text(strip=True) if position_tag else None
    rows.append({
            "stat": f"Primary Position",
            "value": position
        })

    df = pd.DataFrame(rows)

    driver.quit() 
    return df
        

In [53]:
def get_current_season_overview(player_id):
    url = f"https://www.fotmob.com/en/players/{player_id}"

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    # wait for page to load (simple version)
    time.sleep(5)
    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")

    stats_box = soup.find("div", class_="css-4yroh7-StatsContainer elcfuwp1")

    if stats_box is None:
        print("No stats for current season")
        driver.quit()
        return pd.DataFrame()  
    stat_blocks = stats_box.find_all("div", class_="css-48dyyr-StatBox elcfuwp6")
    rows = []
    for block in stat_blocks:
        value_tag = block.find("div", class_="css-170fd60-StatValue elcfuwp5")
        value = value_tag.get_text(strip=True) if value_tag else None
        title_tag = block.find("span", class_="css-1xy07gm-StatTitle elcfuwp4")
        title = title_tag.get_text(strip=True) if title_tag else None
        if title == "Rating":
            rating_block = block.find("div", class_="css-phu8uv-PlayerRatingCSS e1xb9tyd0")
            rating = rating_block.find("span")
            value = rating.get_text(strip=True) if rating else value
        rows.append({
            "stat": title,
            "value": value
        })
    df = pd.DataFrame(rows)
    driver.quit() 
    return df
    

In [54]:
def get_current_season_stats(player_id):
    url = f"https://www.fotmob.com/en/players/{player_id}"

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    wait = WebDriverWait(driver, 10)
    html_no_filters = driver.page_source

    first_stat_value = driver.find_element(
        By.CSS_SELECTOR,
        "div.css-13fpglj-StatValue.e1fqvhy52 span"
    ).text
  
    button = wait.until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, ".css-bmwvkt-FilterButton")
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        button
    )

    button.click()

    # select Per 90
    per90_option = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//*[contains(text(), 'Per 90')]")
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        per90_option
    )

    time.sleep(2)
    wait.until(
        lambda d: d.find_element(
            By.CSS_SELECTOR,
            "div.css-13fpglj-StatValue.e1fqvhy52 span"
        ).text != first_stat_value
    )
    html_with_filters = driver.page_source

    soup_no_filters = BeautifulSoup(html_no_filters, "lxml")
    soup_with_filters = BeautifulSoup(html_with_filters, "lxml")

    rows = []
    for i, soup in enumerate([soup_no_filters, soup_with_filters]):
        block = soup.find("div", class_="css-15lw8xy-SeasonPerformanceCSS e1fqvhy58")
        if block is None:
            print("No season stats available")
            driver.quit()
            return pd.DataFrame()
        stat_items = block.find_all("div", class_="css-1v73fp6-StatItemCSS e1fqvhy50")
        for group in stat_items:
            stat_title_tag = group.find("span", class_="css-1u1ywg2-StatTitle e1fqvhy51")
            stat_title = stat_title_tag.get_text(strip=True) if stat_title_tag else None
            stat_value_tag = group.find("div", class_="css-13fpglj-StatValue e1fqvhy52")
            stat_value = stat_value_tag.get_text(strip=True) if stat_value_tag else None
            if i == 1:  # per 90 stats
                stat_title = f"{stat_title} per 90"
            rows.append({
                "stat": stat_title,
                "value": stat_value
            })

    df = pd.DataFrame(rows)
    driver.quit()
    return df

In [55]:
bio_df = get_player_bio("941576")
season_overview_df = get_current_season_overview("941576")

Leah Williamson


In [56]:
bio_df

,stat,value
0,Height,170 cm
1,Shirt,6
2,"Mar 29, 1997",29 years
3,Preferred foot,Right
4,Country,England
5,Primary Position,Center-back


In [57]:
season_overview_df

,stat,value
0,Goals,1
1,Assists,0
2,Started,2
3,Matches,6
4,Minutes played,233
5,Rating,7.69


In [ ]:
season_stats_df = get_current_season_stats("941576")
season_stats_df

,stat,value
0,Goals,1
1,Expected goals (xG),0.15
2,xG on target (xGOT),0.25
3,Non-penalty xG,0.15
4,Shots,2
...,...,...
61,Clean sheets per 90,0.77
62,Goals conceded while on pitch per 90,0.00
63,xG against while on pitch per 90,0.51
64,Yellow cards per 90,0.39
